# Analisis Sentimen ESG — LinkedIn Perusahaan Tambang Indonesia
---
**Notebook:** 02_sentiment_analysis  
**Deskripsi:** Lexicon-based Sentiment Analysis pada postingan LinkedIn terkait ESG perusahaan tambang.

Pipeline:
1. Load data processed CSV
2. Lexicon-based sentiment (TextBlob + fallback InSet)
3. Visualisasi distribusi, time series, word cloud
4. Export summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from textblob import TextBlob
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Paths
DATA_DIR = Path('../data')
RESULT_DIR = Path('../results')
FIGURE_DIR = Path('../paper/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
# Load processed CSV — update path if needed
csv_files = list(DATA_DIR.glob('processed/*_processed.csv'))
if not csv_files:
    # Fallback: load from original location
    csv_files = list(Path('../../clean').glob('*_processed.csv'))
    
if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f'Loaded: {csv_files[0].name}')
else:
    # Sample data for demonstration
    print('No processed CSV found. Loading sample...')
    import json
    raw_files = list(DATA_DIR.glob('raw/*_offline_raw.json'))
    if raw_files:
        with open(raw_files[0]) as f:
            posts = json.load(f)
        from src.extract import to_row
        rows = [to_row(p) for p in posts]
        df = pd.DataFrame(rows)
    else:
        raise FileNotFoundError('No data files found')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Companies: {df["company_norm"].unique() if "company_norm" in df.columns else df["company"].unique()}')

In [ ]:
df.head(2)

## 2. Lexicon-based Sentiment Analysis

In [ ]:
# TextBlob polarity scorer
def textblob_score(text):
    """Return polarity from TextBlob (-1 to 1)."""
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return TextBlob(text).sentiment.polarity

def classify_sentiment(score):
    """Classify polarity score into positive/negative/neutral."""
    if score > 0.1:
        return 'Positif'
    elif score < -0.1:
        return 'Negatif'
    else:
        return 'Netral'

# Apply to text_raw (mostly bilingual Indo-English)
df['polarity'] = df['text_raw'].apply(textblob_score)
df['sentiment'] = df['polarity'].apply(classify_sentiment)

print(df['sentiment'].value_counts())
print(f"\nMean polarity: {df['polarity'].mean():.3f}")
print(f"Std polarity: {df['polarity'].std():.3f}")

In [ ]:
# Show examples of each sentiment
for sent in ['Positif', 'Netral', 'Negatif']:
    sample = df[df['sentiment'] == sent]['text_raw'].iloc[0] if len(df[df['sentiment'] == sent]) > 0 else 'N/A'
    print(f'=== {sent} ===')
    print(sample[:200])
    print()

## 3. Visualisasi Distribusi Sentimen

In [ ]:
# 3.1 Bar Chart Distribusi Sentimen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sent_counts = df['sentiment'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
axes[0].pie(sent_counts.values, labels=sent_counts.index, autopct='%1.1f%%',
            colors=colors[:len(sent_counts)], startangle=90)
axes[0].set_title('Distribusi Sentimen (Semua Perusahaan)')

# Bar chart per company
if 'company_norm' in df.columns:
    company_col = 'company_norm'
else:
    company_col = 'company'
    
sent_by_company = df.groupby([company_col, 'sentiment']).size().unstack(fill_value=0)
sent_by_company.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Sentimen per Perusahaan')
axes[1].set_xlabel('Perusahaan')
axes[1].set_ylabel('Jumlah Postingan')
axes[1].legend(title='Sentimen')
plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_distribusi_sentimen.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Time Series Sentimen
if 'month' in df.columns:
    ts = df.groupby(['month', 'sentiment']).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ts.plot(kind='line', marker='o', ax=ax, color=colors, linewidth=2)
    ax.set_title('Tren Sentimen per Bulan')
    ax.set_xlabel('Bulan')
    ax.set_ylabel('Jumlah Postingan')
    ax.legend(title='Sentimen')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '02_tren_sentimen.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Column "month" not found — skipping time series')

In [ ]:
# 3.3 Word Cloud (positif vs negatif)
try:
    from wordcloud import WordCloud
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    for idx, sentiment in enumerate(['Positif', 'Negatif']):
        text = ' '.join(df[df['sentiment'] == sentiment]['text_clean'].dropna())
        if text.strip():
            wc = WordCloud(width=800, height=400, background_color='white',
                          colormap='viridis' if sentiment == 'Positif' else 'Reds',
                          max_words=50).generate(text)
            axes[idx].imshow(wc, interpolation='bilinear')
            axes[idx].axis('off')
            axes[idx].set_title(f'Word Cloud — Sentimen {sentiment}', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '03_wordcloud.png', dpi=300, bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed. Install with: pip install wordcloud')
    print('Skipping word cloud...')

In [ ]:
# 3.4 Engagement vs Sentimen
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['reactions_count', 'comments_count', 'reshares_count']
titles = ['Reactions vs Sentimen', 'Comments vs Sentimen', 'Reshares vs Sentimen']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    if metric in df.columns:
        df[metric] = pd.to_numeric(df[metric], errors='coerce').fillna(0)
        sns.boxplot(data=df, x='sentiment', y=metric, ax=axes[idx],
                   palette=colors, order=['Positif', 'Netral', 'Negatif'])
        axes[idx].set_title(title)
        axes[idx].set_ylabel(metric)

plt.tight_layout()
plt.savefig(FIGURE_DIR / '04_engagement_vs_sentimen.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. ESG Category Analysis

In [ ]:
# Categorize posts by ESG pillar
from src.config import categorize_esg

df['esg_pillars'] = df['text_raw'].apply(categorize_esg)
df['esg_pillars_str'] = df['esg_pillars'].apply(lambda x: '/'.join(x) if x else 'None')

# Distribution
esg_dist = df['esg_pillars_str'].value_counts()
print('=== ESG Pillar Distribution ===')
print(esg_dist)

# Sentiment by ESG pillar
fig, ax = plt.subplots(figsize=(10, 5))
if 'company_norm' in df.columns:
    pivot = df.groupby(['company_norm', 'esg_pillars_str']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', ax=ax, edgecolor='white')
    ax.set_title('ESG Pillar Distribution per Company')
    ax.set_xlabel('Company')
    ax.set_ylabel('Post Count')
    ax.legend(title='ESG Pillar')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '05_esg_pillar_dist.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Export Summary

In [ ]:
# Summary statistics
summary = {
    'total_posts': len(df),
    'positive_pct': (df['sentiment'] == 'Positif').mean() * 100,
    'negative_pct': (df['sentiment'] == 'Negatif').mean() * 100,
    'neutral_pct': (df['sentiment'] == 'Netral').mean() * 100,
    'mean_polarity': df['polarity'].mean(),
    'company_count': df['company_norm'].nunique() if 'company_norm' in df.columns else df['company'].nunique(),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULT_DIR / 'sentiment_summary.csv', index=False)
print('=== Sentiment Summary ===')
print(summary_df.T)
print(f'\nSaved to: {RESULT_DIR / "sentiment_summary.csv"}')

In [ ]:
# Per-company breakdown
company_col = 'company_norm' if 'company_norm' in df.columns else 'company'
per_company = df.groupby(company_col).agg(
    total=('sentiment', 'count'),
    positif=('sentiment', lambda x: (x == 'Positif').sum()),
    negatif=('sentiment', lambda x: (x == 'Negatif').sum()),
    netral=('sentiment', lambda x: (x == 'Netral').sum()),
    mean_polarity=('polarity', 'mean'),
    avg_reactions=('reactions_count', 'mean'),
).reset_index()

per_company['positif_pct'] = (per_company['positif'] / per_company['total'] * 100).round(1)
per_company['negatif_pct'] = (per_company['negatif'] / per_company['total'] * 100).round(1)

per_company.to_csv(RESULT_DIR / 'sentiment_per_company.csv', index=False)
print('=== Per-Company Sentiment ===')
display(per_company)
print(f'\nSaved to: {RESULT_DIR / "sentiment_per_company.csv"}')

In [ ]:
# Extreme sentiment posts (for thematic analysis)
top_positif = df.nlargest(50, 'polarity')
top_negatif = df.nsmallest(50, 'polarity')

extreme = pd.concat([
    top_positif[['company_norm' if 'company_norm' in df.columns else 'company', 'post_id', 'text_raw', 'polarity']],
    top_negatif[['company_norm' if 'company_norm' in df.columns else 'company', 'post_id', 'text_raw', 'polarity']]
])

extreme.to_csv(RESULT_DIR / 'extreme_sentiment_posts.csv', index=False)
print(f'Extreme posts for thematic analysis: {len(extreme)}')
print(f'Saved to: {RESULT_DIR / "extreme_sentiment_posts.csv"}')

## 6. Kesimpulan

Notebook ini menyediakan:
1. **Sentiment distribution** — visualisasi pie, bar, time series
2. **Word cloud** — kata dominan per sentimen
3. **Engagement analysis** — hubungan reactions/comments/reshares dengan sentimen
4. **ESG pillar categorization** — distribusi E, S, G per perusahaan
5. **Exported outputs** — CSV summary + extreme posts untuk thematic analysis

Hasil ini menjadi input untuk **Fase Kualitatif** (*Reflexive Thematic Analysis*) pada notebook 03_thematic_analysis.ipynb.